# 🧪 Caderno 01: Análise Exploratória e Pipeline de Pré-processamento Espectral
**Projeto:** Detecção de Adulteração em Fertilizantes NPK via LIBS  
**Instituição:** Embrapa Instrumentação  

---

### Objetivos deste Caderno:
1. **Carregamento dos dados brutos** para os canais ópticos **UV** e **VIS**.
2. **Avaliação comparativa de filtros de ruído:** Espectro Bruto vs Wavelet (*sym8*) vs Savitzky-Golay.
3. **Avaliação da remoção de linha de base:** Filtro morfológico *White Top-Hat*
4. **Normalização espectral:** *Standard Normal Variate (SNV)* e Normalização por Área (Regra de Simpson).
5. **Filtragem de disparos anômalos (outliers):** Algoritmo *Spectral Angle Mapper (SAM)*.
6. **Inspeção dos espectros médios consolidados** por teor de adulteração (0%, 10%, 25%, 50%).

In [ ]:
# Adiciona a pasta raiz ao sys.path para importar o pacote src
import sys
import os
from pathlib import Path

# Ajuste do caminho base
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importação dos módulos especializados
from src.preprocessing import (
    carregar_diretorio_libs,
    carregar_esf_bruto,
    wavelet_denoising,
    savgol_denoising,
    tophat_baseline_correction,
    iterative_polynomial_baseline,
    snv_normalizacao,
    normalizacao_area,
    filtrar_outliers_sam,
    processar_espectro,
    processar_lote_espectros,
)
from src.visualization import (
    plotar_comparacao_filtros_ruido,
    plotar_espectros_medios
)

print("Módulos carregados com sucesso!")

## 1. Carregamento dos Arquivos Brutos (.esf)
Os arquivos espectrais devem estar organizados em pastas contendo os teores de adulteração (`00porcento`, `10porcento`, `25porcento`, `50porcento`) e subpastas para os canais (`UV` e `VIS`).

In [ ]:
# Defina o diretório onde estão os espectros brutos
# Exemplo local: BASE_DIR / 'data' / 'raw' ou caminho personalizado
CAMINHO_DADOS = BASE_DIR / "data" / "raw"

# Caso esteja executando no Google Colab, descomente e ajuste:
# CAMINHO_DADOS = Path("/content/drive/MyDrive/LIBS/Espectros/NPK_10-10-10_Fabricado")

print(f"Diretório de dados configurado: {CAMINHO_DADOS}")

# Carregamento em lote dos metadados e matrizes brutas
if CAMINHO_DADOS.exists():
    lista_uv_bruto, lista_vis_bruto = carregar_diretorio_libs(CAMINHO_DADOS)
else:
    print(f"Aviso: Diretório {CAMINHO_DADOS} não encontrado. Crie a pasta ou configure o caminho correto.")
    lista_uv_bruto, lista_vis_bruto = [], []

## 2. Análise do Tratamento de Ruído (Denoising)
Comparação entre o sinal bruto e dois dos filtros mais empregados em LIBS:
- **Transformada Wavelet Discreta (DWT):** Família Symlet 8 (`sym8`) com limiarização adaptativa de Donoho-Johnstone (MAD).
- **Savitzky-Golay:** Suavização polinomial local de 2ª ordem.

In [ ]:
if len(lista_vis_bruto) > 0:
    amostra_exemplo = lista_vis_bruto[0]
    wl = amostra_exemplo['Wavelength']
    sinal_bruto = amostra_exemplo['Intensity_Raw']

    sinal_wavelet = wavelet_denoising(sinal_bruto, wavelet='sym8', level=6)
    sinal_savgol = savgol_denoising(sinal_bruto, window_length=7, polyorder=2)

    fig = plotar_comparacao_filtros_ruido(
        wl, sinal_bruto, sinal_wavelet, sinal_savgol,
        titulo=f"Comparação de Filtros de Ruído - {amostra_exemplo['Nome_Arquivo']}"
    )
    plt.show()

## 3. Análise da Remoção de Linha de Base (Baseline)
O plasma gerado por laser produz um fundo contínuo de radiação bremsstrahlung e recombinação.
Testamos aqui o operador morfológico **White Top-Hat** (com elemento estruturante de 100 pontos) comparado ao **Ajuste Polinomial Iterativo (ModPoly)**.

In [ ]:
if len(lista_vis_bruto) > 0:
    sinal_tophat = tophat_baseline_correction(sinal_bruto, footprint_size=100)
    baseline_poly, sinal_poly = iterative_polynomial_baseline(wl, sinal_bruto, order=4)

    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    
    # White Top-Hat
    axes[0].plot(wl, sinal_bruto, label="Sinal Bruto", color="gray", alpha=0.6)
    axes[0].plot(wl, sinal_tophat, label="Pós White Top-Hat (Filtro Morfológico)", color="tab:blue")
    axes[0].set_title("Correção com White Top-Hat", fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.4)

    # Polinomial Iterativo
    axes[1].plot(wl, sinal_bruto, label="Sinal Bruto", color="gray", alpha=0.6)
    axes[1].plot(wl, baseline_poly, label="Linha de Base Estimada", color="tab:red", linestyle="--")
    axes[1].plot(wl, sinal_poly, label="Pós Subtração Polinomial", color="tab:green")
    axes[1].set_title("Correção Polinomial Iterativa", fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.4)

    plt.xlabel("Comprimento de Onda (nm)")
    plt.tight_layout()
    plt.show()

## 4. Normalização Espectral e Filtragem de Outliers (SAM)
Aplicamos o pipeline de processamento completo em lote:
1. **White Top-Hat** (Baseline)
2. **SNV** (Standard Normal Variate) + deslocamento ao zero
3. **Filtragem de Outliers via SAM (Spectral Angle Mapper)** com corte no percentil 99.

In [ ]:
# Processamento em lote para o Canal VIS
if len(lista_vis_bruto) > 0:
    print("Processando canal VIS...")
    df_processado_vis = processar_lote_espectros(
        lista_vis_bruto,
        baseline_footprint=100,
        norm_method='snv',
        subtrair_minimo=True
    )
    print(f"Dimensões VIS antes do SAM: {df_processado_vis.shape}")

    # Filtragem de outliers via SAM
    df_limpo_vis = filtrar_outliers_sam(df_processado_vis, threshold_percentil=99.0)
    print(f"Dimensões VIS após SAM: {df_limpo_vis.shape}")
else:
    print("Sem dados brutos carregados para processamento.")

In [ ]:
# Processamento em lote para o Canal UV
if len(lista_uv_bruto) > 0:
    print("Processando canal UV...")
    df_processado_uv = processar_lote_espectros(
        lista_uv_bruto,
        baseline_footprint=100,
        norm_method='snv',
        subtrair_minimo=True
    )
    print(f"Dimensões UV antes do SAM: {df_processado_uv.shape}")

    df_limpo_uv = filtrar_outliers_sam(df_processado_uv, threshold_percentil=99.0)
    print(f"Dimensões UV após SAM: {df_limpo_uv.shape}")

## 5. Visualização dos Espectros Médios por Teor de Adulteração
Inspecionamos os perfis médios obtidos para cada classe (0% Puro vs 10%, 25%, 50% Adulterado).

In [ ]:
if 'df_limpo_vis' in locals() and len(df_limpo_vis) > 0:
    fig_vis = plotar_espectros_medios(
        df_limpo_vis,
        titulo="Espectros Médios Normalizados (SNV) - Canal VIS"
    )
    plt.show()

if 'df_limpo_uv' in locals() and len(df_limpo_uv) > 0:
    fig_uv = plotar_espectros_medios(
        df_limpo_uv,
        titulo="Espectros Médios Normalizados (SNV) - Canal UV"
    )
    plt.show()

## 6. Salvamento das Matrizes Tratadas
Para evitar ter que reprocessar arquivos `.esf` a cada análise, salvamos os dados tratados em formato Parquet/CSV na pasta `data/processed`.

In [ ]:
pasta_saida = BASE_DIR / "data" / "processed"
pasta_saida.mkdir(parents=True, exist_ok=True)

if 'df_limpo_vis' in locals() and len(df_limpo_vis) > 0:
    caminho_vis = pasta_saida / "df_limpo_vis.parquet"
    df_limpo_vis.to_parquet(caminho_vis, index=False)
    print(f"VIS salvo em: {caminho_vis}")

if 'df_limpo_uv' in locals() and len(df_limpo_uv) > 0:
    caminho_uv = pasta_saida / "df_limpo_uv.parquet"
    df_limpo_uv.to_parquet(caminho_uv, index=False)
    print(f"UV salvo em: {caminho_uv}")